In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from deeporigin.drug_discovery import  BRD_DATA_DIR, Protein, Ligand, LigandSet, ABFE, SystemPrep, ABFEParams, PreparedSystem
from deeporigin.platform import DeepOriginClient

client = DeepOriginClient()


# ABFE workflow

This notebook shows you how to run ABFE on Deep Origin, and serves as a quick tutorial for running ABFE. 


In [ ]:
ligands = LigandSet.from_dir(BRD_DATA_DIR)
ligands.to_dataframe()

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.sync()
protein.show()

In [ ]:
# use this ligand
ligand = [ligand for ligand in ligands if ligand.name == "cmpd 4 (Crotyl)"][0]
ligand.sync()
ligand

## Prepare system

In [ ]:
# see if we have prepared systems for this. if now, prepare one
systems = PreparedSystem.from_result(protein_id=protein.id)
sp = SystemPrep(protein=protein, ligand=ligand)

if len(systems) > 0:
    system = systems[0]
else:
    system = sp.run()
    
system.show()



## Quote ABFE

We can estimate the cost of an ABFE run without running it. 

In [ ]:
abfe = ABFE(prepared_system=system, params=ABFEParams(test_run=0))


In [ ]:
abfe

In [ ]:
abfe.quote()
abfe.estimate

If we're happy with this price, we can confirm the job (which runs it)

In [ ]:
abfe.start()

In [ ]:
task = await abfe.watch()

## Monitor job

To monitor a job, use the job.watch method:

## Get results

In [ ]:
abfe.get_results()


In [ ]:
abfe.sync()
abfe.progress